# jC_03 - Far-domain x rasio serangan (JURNAL, Tahap 1B)

**Pertanyaan:** dengan 9 fitur yang sama, far-domain (CIC+UNSW) mencapai MCC 0,647 sedangkan near-domain
(CIC2017+CIC2018) hanya 0,537. Naskah menduga UNSW **bermayoritas serangan (55%) dan lebih mudah dipisah**,
sehingga far-domain tampak lebih baik. Dugaan ini belum diuji.

**Cara menguji:** jalankan federasi far-domain (FedAvg, 100 ronde, 5 seed, protokol bebas-kebocoran) dalam dua kondisi:
- **A (asli)**: rasio serangan asli. Sekaligus *uji kewajaran*: hasil simulasi ini harus dekat 0,647 (hasil EC2).
- **B (UNSW 17%)**: UNSW diturunkan ke 17% serangan (train dan test), sehingga setara rasio near-domain (19-24%).
Kondisi B juga dievaluasi pada test set UNSW **asli** sebagai pembanding (kolom `orig`).

**Kriteria dibaca (disepakati SEBELUM melihat angka):**
- Uji kewajaran: A harus di 0,627-0,667 (CI hasil EC2). Bila jauh, simulasi vs EC2 berbeda dan hasil B harus dibaca hati-hati.
- Dugaan **didukung** bila B (matched) <= 0,58 (turun mendekati near@9 = 0,537).
- **Tidak didukung** bila B (matched) berada dalam +/-0,03 dari A.
- Di antaranya = sebagian.

Catatan: MCC bergantung pada prevalensi kelas; karena itu B dievaluasi pada test set dengan rasio sama (matched) dan pada test asli.

**Prasyarat:** data far-domain hasil s01 di S3. **Output:** `journal_out/r3ratio_far_{seeds,agg}.csv` + S3
`federated_journal/ratio/results/`. Jalankan di SageMaker.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','boto3','scipy','pandas'):
    if importlib.util.find_spec(m) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, csv, time, numpy as np, pandas as pd, boto3

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
SRC='unsw-far/federated_survey'      # data far-domain 9 fitur (hasil s01), sudah ada di S3
OUT='federated_journal/ratio'        # prefix hasil BARU untuk percobaan rasio serangan
OUTDIR='journal_out'; os.makedirs(OUTDIR, exist_ok=True)
s3=boto3.client('s3',region_name=REGION)

def s3_get_npz(key):
    obj=s3.get_object(Bucket=BUCKET,Key=key)
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

def load_raw():
    raw={}
    for ds in ('cic','unsw'):
        raw[ds]={}
        for sp in ('train','test'):
            d=s3_get_npz(f'{SRC}/data/{ds}/{ds}_{sp}.npz')
            raw[ds][sp]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
        print(ds,'train',raw[ds]['train'][0].shape,'pos=%.3f'%raw[ds]['train'][1].mean(),
              '| test',raw[ds]['test'][0].shape,'pos=%.3f'%raw[ds]['test'][1].mean())
    return raw
RAW=load_raw()

In [ ]:
HID=[128,64]
def init_w(d, seed=42):
    rng=np.random.RandomState(seed); W=[]; L=[d]+HID+[1]
    for a,b in zip(L[:-1],L[1:]):
        W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]
    return _sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs,lr,seed,batch=256):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/max(len(yb),1)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0)
            gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def fedavg(Ws,ns):
    tot=float(sum(ns)); return [sum((ns[k]/tot)*Ws[k][i] for k in range(len(Ws))) for i in range(len(Ws[0]))]
def zfit(X):
    m=X.mean(0); s=X.std(0); s[s==0]=1.0; return m,s
def mcc(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); tn=int(((p==0)&(y==0)).sum())
    fp=int(((p==1)&(y==0)).sum()); fn=int(((p==0)&(y==1)).sum())
    den=np.sqrt(float(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den>0 else 0.0
def recall(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); fn=int(((p==0)&(y==1)).sum())
    return tp/(tp+fn) if (tp+fn)>0 else 0.0
def dirichlet_partition(y,K,alpha,seed):
    rng=np.random.RandomState(seed); client_idx=[[] for _ in range(K)]
    for c in np.unique(y):
        idc=np.where(y==c)[0]; rng.shuffle(idc)
        props=rng.dirichlet([alpha]*K); cuts=(np.cumsum(props)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc,cuts)): client_idx[k].extend(part.tolist())
    return [np.array(c) for c in client_idx if len(c)>0]
def resample_to_ratio(X,y,r,seed=0):
    """Ubah rasio serangan ke r dgn membuang sampel kelas yang berlebih (data tetap & deterministik)."""
    rng=np.random.RandomState(seed); pos=np.where(y==1)[0]; neg=np.where(y==0)[0]
    cur=len(pos)/len(y)
    if abs(cur-r)<1e-3: return X,y
    if cur>r:
        keep=int(round(len(neg)*r/(1-r))); pos=rng.choice(pos,size=min(keep,len(pos)),replace=False)
    else:
        keep=int(round(len(pos)*(1-r)/r)); neg=rng.choice(neg,size=min(keep,len(neg)),replace=False)
    idx=np.concatenate([pos,neg]); rng.shuffle(idx); return X[idx],y[idx]
print('util siap')

In [ ]:
CONDS={'A: asli':None,'B: UNSW 17%':0.17}
def build(cond):
    r=CONDS[cond]; C={}
    for c in ('cic','unsw'):
        Xtr,ytr=RAW[c]['train']; Xte,yte=RAW[c]['test']; Xo,yo=Xte,yte
        if r is not None and c=='unsw':
            Xtr,ytr=resample_to_ratio(Xtr,ytr,r); Xte,yte=resample_to_ratio(Xte,yte,r)
        C[c]=dict(Xtr=Xtr,ytr=ytr,matched=(Xte,yte),orig=(Xo,yo))
    return C
print('%-14s %-5s %8s %8s | %8s %8s'%('kondisi','klien','n_train','pos_tr','n_test','pos_te'))
for cond in CONDS:
    C=build(cond)
    for c in C: print('%-14s %-5s %8d %8.3f | %8d %8.3f'%(cond,c,len(C[c]['ytr']),C[c]['ytr'].mean(),
                                                         len(C[c]['matched'][1]),C[c]['matched'][1].mean()))

In [ ]:
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; SEEDS=[13,42,101,202,303]

def run_fed(cond, seed):
    C=build(cond); zst={c:zfit(C[c]['Xtr']) for c in C}   # z-score LOKAL per client (bebas-kebocoran)
    Xz={c:(C[c]['Xtr']-zst[c][0])/zst[c][1] for c in C}
    d=C['cic']['Xtr'].shape[1]; W=init_w(d,seed)
    for t in range(ROUNDS):
        Ws=[]; ns=[]
        for c in C:
            Ws.append(train_local(W,Xz[c],C[c]['ytr'],LOCAL_EPOCHS,LR,seed)); ns.append(len(C[c]['ytr']))
        W=fedavg(Ws,ns)
    rows=[]
    for tag in ('matched','orig'):
        pa=[]; ya=[]
        for c in C:
            X,y=C[c][tag]; o,_=fwd(W,(X-zst[c][0])/zst[c][1]); p=(o>=0.5).astype(int)
            rows.append(dict(cond=cond,seed=seed,scope=c,test=tag,mcc=mcc(y,p),recall=recall(y,p)))
            pa.append(p); ya.append(y)
        pa=np.concatenate(pa); ya=np.concatenate(ya)
        rows.append(dict(cond=cond,seed=seed,scope='union',test=tag,mcc=mcc(ya,pa),recall=recall(ya,pa)))
    return rows

SEEDROWS=[]
for cond in CONDS:
    for sd in SEEDS:
        t0=time.time(); r=run_fed(cond,sd); SEEDROWS+=r
        u=[x for x in r if x['scope']=='union' and x['test']=='matched'][0]['mcc']
        print(f'{cond} seed={sd:<3} union MCC (matched)={u:+.3f}  [{time.time()-t0:.0f}s]')
sp=os.path.join(OUTDIR,'r3ratio_far_seeds.csv')
pd.DataFrame(SEEDROWS).to_csv(sp,index=False); s3.upload_file(sp,BUCKET,f'{OUT}/results/r3ratio_far_seeds.csv')
print('CSV per seed tersimpan + upload')

In [ ]:
from scipy import stats
df=pd.DataFrame(SEEDROWS)
def mci(v):
    a=np.asarray(v,float); n=len(a); h=a.std(ddof=1)/np.sqrt(n)*stats.t.ppf(0.975,n-1) if n>1 else 0.0
    return a.mean(),h
agg=[]
for (cond,scope,test),g in df.groupby(['cond','scope','test']):
    m,h=mci(g.mcc); r_,hr=mci(g.recall)
    agg.append(dict(cond=cond,scope=scope,test=test,mcc_mean=round(m,4),mcc_ci95=round(h,4),
                    recall_mean=round(r_,4),n=len(g)))
agg=pd.DataFrame(agg); ap=os.path.join(OUTDIR,'r3ratio_far_agg.csv')
agg.to_csv(ap,index=False); s3.upload_file(ap,BUCKET,f'{OUT}/results/r3ratio_far_agg.csv')
print('=== UNION MCC (mean +/- CI95) ===')
for _,r in agg[agg.scope=='union'].sort_values(['cond','test']).iterrows():
    print(f"  {r.cond:12s} test={r.test:8s} {r.mcc_mean:+.3f} +/- {r.mcc_ci95:.3f}")
A=agg[(agg.cond=='A: asli')&(agg.scope=='union')&(agg.test=='matched')].iloc[0].mcc_mean
B=agg[(agg.cond=='B: UNSW 17%')&(agg.scope=='union')&(agg.test=='matched')].iloc[0].mcc_mean
print('\nKriteria (disepakati sebelum melihat angka):')
print(f'  Uji kewajaran: A={A:.3f} berada di 0.627-0.667 ?', 0.627<=A<=0.667)
print(f'  B (matched)={B:.3f} <= 0.58 (mendukung) ?', B<=0.58)
print(f'  |B-A|={abs(B-A):.3f} <= 0.03 (tidak mendukung) ?', abs(B-A)<=0.03)
print('Jangan menyimpulkan sebelum Kiro membaca hasilnya.')

In [ ]:
NEED=[f'{OUT}/results/r3ratio_far_seeds.csv',f'{OUT}/results/r3ratio_far_agg.csv']
resp=s3.list_objects_v2(Bucket=BUCKET,Prefix=f'{OUT}/')
keys=sorted(o['Key'] for o in resp.get('Contents',[]))
print('File hasil di S3 (%d):'%len(keys))
for k in keys: print('  s3://%s/%s'%(BUCKET,k))
miss=[k for k in NEED if k not in keys]
print('LENGKAP' if not miss else 'KURANG: %s'%miss)
print()
print('Di komputer lokal (PowerShell), tarik dengan:')
print('  aws s3 sync s3://%s/%s/ ./ratio'%(BUCKET,OUT))